# 실습 기자재 YOLO 학습 및 Raspberry Pi 배포

이 노트북은 기자재 데이터셋 확인, nano 모델 학습, 검증, 테스트 이미지 추론, NCNN 변환까지 수행합니다. 처음에는 3~5개 클래스와 입력 크기 416으로 시작하세요.

In [ ]:
!pip install -q -U ultralytics
from ultralytics import YOLO
import ultralytics
ultralytics.checks()

## 데이터셋 연결

Google Drive에 `equipment_dataset` 폴더를 올려 둔 경우 아래 셀을 실행합니다. `data.yaml` 안의 path도 실제 위치에 맞게 수정하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_YAML = '/content/drive/MyDrive/equipment_dataset/data.yaml'
BASE_MODEL = 'yolo11n.pt'  # nano 모델. 필요하면 지원되는 최신 nano 모델로 변경
RUN_NAME = 'equipment_yolo_nano'

In [ ]:
from pathlib import Path

yaml_path = Path(DATA_YAML)
assert yaml_path.exists(), f'data.yaml을 찾을 수 없습니다: {yaml_path}'
print(yaml_path.read_text(encoding='utf-8'))

## 학습

첫 학습은 80 epoch로 시작하고 결과 그래프를 확인합니다. Colab GPU 메모리가 부족하면 batch를 8로 낮추세요.

In [ ]:
model = YOLO(BASE_MODEL)
train_results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=416,
    batch=16,
    patience=15,
    device=0,
    workers=2,
    project='/content/runs/equipment',
    name=RUN_NAME,
)

## 검증 및 클래스 이름 확인

정확도 숫자만 보지 말고 confusion matrix와 실제 오인식 이미지를 함께 확인하세요.

In [ ]:
BEST_PT = f'/content/runs/equipment/{RUN_NAME}/weights/best.pt'
best_model = YOLO(BEST_PT)
metrics = best_model.val(data=DATA_YAML, imgsz=416)
print('클래스:', best_model.names)
print('mAP50:', float(metrics.box.map50))
print('mAP50-95:', float(metrics.box.map))

## 테스트 이미지 추론

학습에 사용하지 않은 실제 라즈베리파이 카메라 사진 폴더를 지정하세요.

In [ ]:
TEST_IMAGES = '/content/drive/MyDrive/equipment_dataset/field_test_images'
predictions = best_model.predict(
    source=TEST_IMAGES,
    imgsz=416,
    conf=0.60,
    save=True,
    project='/content/runs/equipment',
    name='field_test',
)
print('테스트 이미지 수:', len(predictions))

## Raspberry Pi용 NCNN 변환

먼저 Pi에서 best.pt가 정상 동작하는지 확인한 다음 NCNN 모델과 속도·정확도를 비교하세요.

In [ ]:
exported_path = best_model.export(format='ncnn', imgsz=320)
print('NCNN 경로:', exported_path)

In [ ]:
from google.colab import files
import shutil

shutil.copy2(BEST_PT, '/content/best.pt')
files.download('/content/best.pt')

archive = shutil.make_archive('/content/best_ncnn_model', 'zip', root_dir=str(exported_path))
files.download(archive)